In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [ ]:
# pip install "transformers>=4.40.0" "huggingface_hub>=0.20.0"

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, PeftModel, TaskType
from datasets import Dataset
from trl import SFTTrainer, SFTConfig

/home/oncreative/.local/lib/python3.10/site-packages/librosa/util/files.py:10: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename
/home/oncreative/.local/lib/python3.10/site-packages/torch/cuda/__init__.py:827: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


In [ ]:
# !pip install trl

In [ ]:
# pip install transformers==4.49

In [ ]:
from trl import SFTTrainer, SFTConfig

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, PeftModel, TaskType
from datasets import Dataset

In [ ]:
MODEL_NAME = 'google/gemma-2-2b-it'


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [ ]:
tokenizer.vocab_size

256000

In [ ]:
train_data = [
    {"text": "<start_of_turn>user\n서울의 유명한 음식은?<end_of_turn>\n<start_of_turn>model\n서울에서 꼭 먹어봐야 할 음식으로는 비빔밥, 떡볶이, 냉면, 삼겹살, 김치찌개 등이 있습니다.<end_of_turn>"},
    {"text": "<start_of_turn>user\n파이썬이 뭐야?<end_of_turn>\n<start_of_turn>model\n파이썬(Python)은 읽기 쉽고 다재다능한 프로그래밍 언어입니다. 웹, 데이터 분석, AI 등에 사용됩니다.<end_of_turn>"},
    {"text": "<start_of_turn>user\n머신러닝과 딥러닝의 차이는?<end_of_turn>\n<start_of_turn>model\n머신러닝은 데이터에서 패턴을 학습하는 AI 분야이고, 딥러닝은 심층 신경망을 사용하는 머신러닝의 하위 분야입니다.<end_of_turn>"},
    {"text": "<start_of_turn>user\nTransformer가 뭐야?<end_of_turn>\n<start_of_turn>model\nTransformer는 Self-Attention 메커니즘 기반의 신경망 아키텍처입니다. GPT, BERT 등 현대 NLP 모델의 기반입니다.<end_of_turn>"},
    {"text": "<start_of_turn>user\nLoRA가 뭐야?<end_of_turn>\n<start_of_turn>model\nLoRA는 사전학습된 가중치를 고정하고 저랭크 행렬로 효율적으로 미세조정하는 PEFT 기법입니다.<end_of_turn>"},
    {"text": "<start_of_turn>user\n한국의 계절은?<end_of_turn>\n<start_of_turn>model\n한국은 봄, 여름, 가을, 겨울 사계절이 뚜렷합니다. 봄에는 벚꽃이, 가을에는 단풍이 아름답습니다.<end_of_turn>"},
    {"text": "<start_of_turn>user\nGPU가 왜 필요해?<end_of_turn>\n<start_of_turn>model\nGPU는 병렬 연산에 특화되어 딥러닝의 행렬 연산을 빠르게 처리합니다. CPU보다 수십~수백 배 빠릅니다.<end_of_turn>"},
    {"text": "<start_of_turn>user\n좋은 코드란?<end_of_turn>\n<start_of_turn>model\n좋은 코드는 읽기 쉽고, 유지보수하기 편하며, 적절한 테스트가 포함된 코드입니다.<end_of_turn>"},
]

In [ ]:
dataset = Dataset.from_list(train_data)

In [ ]:
len(dataset)

8

In [ ]:
device = 'cuda'

In [ ]:
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16).to(device)
model

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

Gemma2ForCausalLM(
  (model): Gemma2Model(
    (embed_tokens): Embedding(256000, 2304, padding_idx=0)
    (layers): ModuleList(
      (0-25): 26 x Gemma2DecoderLayer(
        (self_attn): Gemma2Attention(
          (q_proj): Linear(in_features=2304, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2304, out_features=1024, bias=False)
          (v_proj): Linear(in_features=2304, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2304, bias=False)
        )
        (mlp): Gemma2MLP(
          (gate_proj): Linear(in_features=2304, out_features=9216, bias=False)
          (up_proj): Linear(in_features=2304, out_features=9216, bias=False)
          (down_proj): Linear(in_features=9216, out_features=2304, bias=False)
          (act_fn): GELUTanh()
        )
        (input_layernorm): Gemma2RMSNorm((2304,), eps=1e-06)
        (post_attention_layernorm): Gemma2RMSNorm((2304,), eps=1e-06)
        (pre_feedforward_layernorm): Gemma2RMSNo

In [ ]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules = ["q_proj", "v_proj"],
    lora_dropout = 0.1,
    bias='none',
    task_type = TaskType.CAUSAL_LM
)

In [ ]:
model = get_peft_model(model, lora_config)

In [ ]:
model.print_trainable_parameters()

trainable params: 1,597,440 || all params: 2,615,939,328 || trainable%: 0.0611


In [ ]:
# model

In [ ]:
model.model.model.layers[0].self_attn.q_proj

lora.Linear(
  (base_layer): Linear(in_features=2304, out_features=2048, bias=False)
  (lora_dropout): ModuleDict(
    (default): Dropout(p=0.1, inplace=False)
  )
  (lora_A): ModuleDict(
    (default): Linear(in_features=2304, out_features=8, bias=False)
  )
  (lora_B): ModuleDict(
    (default): Linear(in_features=8, out_features=2048, bias=False)
  )
  (lora_embedding_A): ParameterDict()
  (lora_embedding_B): ParameterDict()
  (lora_magnitude_vector): ModuleDict()
)

In [ ]:
model.model.model.layers[0].self_attn.q_proj.base_layer.weight.shape

torch.Size([2048, 2304])

In [ ]:
model.model.model.layers[0].self_attn.q_proj.base_layer.weight.requires_grad

False

In [ ]:
model.model.model.layers[0].self_attn.q_proj.lora_A['default'].weight.shape

torch.Size([8, 2304])

In [ ]:
model.model.model.layers[0].self_attn.q_proj.lora_A['default'].weight.requires_grad

True

In [ ]:
model.model.model.layers[0].self_attn.q_proj.lora_B['default'].weight.shape

torch.Size([2048, 8])

In [ ]:
model.model.model.layers[0].self_attn.q_proj.lora_B['default'].weight.requires_grad

True

In [ ]:
def test_generate(model, prompt):
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=80, do_sample=False)

    print(f"input : {prompt}")
    print(f"output : {tokenizer.decode(out[0], skip_special_tokens=True)}")
    # return tokenizer.decode(out[0], skip_special_tokens=True)

In [ ]:
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16).to(device)
test_generate(model, "한국의 수도는")

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

input : 한국의 수도는
output : 한국의 수도는 서울입니다.

**서울**은 대한민국에서 가장 큰 도시 중 하나이며, 역사와 문화, 그리고 현대적인 도시의 조화를 이루는 도시입니다. 

**서울의 주요 관광 명소**

* **경복궁:** 조선시대의 대표적인 왕궁입니다.
* **북촌:** 전통적인 한옥


In [ ]:
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16).to(device)
model = get_peft_model(model, lora_config)

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

In [ ]:
args = SFTConfig(
    output_dir = './lora_output',
    num_train_epochs = 10,
    per_device_train_batch_size=2,
    learning_rate = 2e-4,
    fp16=True
)

trainer = SFTTrainer(
    model = model,
    args = args,
    train_dataset = dataset,
    processing_class = tokenizer)

Adding EOS to train dataset:   0%|          | 0/8 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/8 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/8 [00:00<?, ? examples/s]

In [ ]:
trainer.train()

Step,Training Loss
10,1.671797
20,1.009198
30,0.698666
40,0.546849


TrainOutput(global_step=40, training_loss=0.9816272854804993, metrics={'train_runtime': 11.0266, 'train_samples_per_second': 7.255, 'train_steps_per_second': 3.628, 'total_flos': 57306645937152.0, 'train_loss': 0.9816272854804993})

In [ ]:
test_generate(model, "한국의 수도는")

input : 한국의 수도는
output : 한국의 수도는????????????????????????????????????????????????????????????????????????????????


In [ ]:
config1 = LoraConfig(
    r= 8,
    lora_alpha = 8,
    target_modules = ['q_proj', 'v_proj', 'v_proj', 'o_proj'],
    task_type= TaskType.CAUSAL_LM
)

In [ ]:
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16).to(device)
model

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

Gemma2ForCausalLM(
  (model): Gemma2Model(
    (embed_tokens): Embedding(256000, 2304, padding_idx=0)
    (layers): ModuleList(
      (0-25): 26 x Gemma2DecoderLayer(
        (self_attn): Gemma2Attention(
          (q_proj): Linear(in_features=2304, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2304, out_features=1024, bias=False)
          (v_proj): Linear(in_features=2304, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2304, bias=False)
        )
        (mlp): Gemma2MLP(
          (gate_proj): Linear(in_features=2304, out_features=9216, bias=False)
          (up_proj): Linear(in_features=2304, out_features=9216, bias=False)
          (down_proj): Linear(in_features=9216, out_features=2304, bias=False)
          (act_fn): GELUTanh()
        )
        (input_layernorm): Gemma2RMSNorm((2304,), eps=1e-06)
        (post_attention_layernorm): Gemma2RMSNorm((2304,), eps=1e-06)
        (pre_feedforward_layernorm): Gemma2RMSNo

In [ ]:
model = get_peft_model(model, config1)
model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Gemma2ForCausalLM(
      (model): Gemma2Model(
        (embed_tokens): Embedding(256000, 2304, padding_idx=0)
        (layers): ModuleList(
          (0-25): 26 x Gemma2DecoderLayer(
            (self_attn): Gemma2Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=2304, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2304, out_features=4, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=4, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): Linear(in_feat

In [ ]:
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16)
cfg = LoraConfig(r=8, lora_alpha=16, target_modules=['q_proj', 'v_proj'], task_type=TaskType.CAUSAL_LM)
model = get_peft_model(model, cfg)

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

In [ ]:
model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Gemma2ForCausalLM(
      (model): Gemma2Model(
        (embed_tokens): Embedding(256000, 2304, padding_idx=0)
        (layers): ModuleList(
          (0-25): 26 x Gemma2DecoderLayer(
            (self_attn): Gemma2Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=2304, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2304, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): Linear(in_feat

In [ ]:
r =8
num_layers = 26
q_A = 2304 * r
q_B = r * 2048
query_layer = q_A + q_B

In [ ]:
v_A = 2304 * r
v_B = r * 1024
value_layer = v_A + v_B

total_lora = (query_layer + value_layer) * num_layers
total_lora

1597440

In [ ]:
model.print_trainable_parameters()

trainable params: 1,597,440 || all params: 2,615,939,328 || trainable%: 0.0611


In [ ]:
fp32 (32bit : 4byte)  -  fp16 (16bit : 2 byte) - int8 (8bit : 1byte) - int4 (4bit : 0.5byte)

In [ ]:
gemma-2-2b-it : 10gb          5gb                    2.5gb                  1.3gb

In [ ]:
QLoRA :
1. NF4 (Normalized Float 4bit)
-
2. scale 도 quantization함


3. gpu 메모리가 부족하면, cpu를 써서 학습

In [ ]:
-3.1122334  -3.1121334 -3.1121333, ..                  1.01      1.23345 )  * 0.01
-128        -127, -110                                              127  )  * 0.01

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit = True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype = torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

In [ ]:
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config = bnb_config, torch_dtype=torch.float16)

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

In [ ]:
model

Gemma2ForCausalLM(
  (model): Gemma2Model(
    (embed_tokens): Embedding(256000, 2304, padding_idx=0)
    (layers): ModuleList(
      (0-25): 26 x Gemma2DecoderLayer(
        (self_attn): Gemma2Attention(
          (q_proj): Linear4bit(in_features=2304, out_features=2048, bias=False)
          (k_proj): Linear4bit(in_features=2304, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=2304, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=2048, out_features=2304, bias=False)
        )
        (mlp): Gemma2MLP(
          (gate_proj): Linear4bit(in_features=2304, out_features=9216, bias=False)
          (up_proj): Linear4bit(in_features=2304, out_features=9216, bias=False)
          (down_proj): Linear4bit(in_features=9216, out_features=2304, bias=False)
          (act_fn): GELUTanh()
        )
        (input_layernorm): Gemma2RMSNorm((2304,), eps=1e-06)
        (post_attention_layernorm): Gemma2RMSNorm((2304,), eps=1e-06)
        (pre_feedfor

In [ ]:
lora_config = LoraConfig(
    r=16, lora_alpha=32, target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'], task_type=TaskType.CAUSAL_LM)

model = get_peft_model(model, lora_config)
model

/home/oncreative/anaconda3/envs/codeit/lib/python3.10/site-packages/peft/mapping_func.py:73: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/home/oncreative/anaconda3/envs/codeit/lib/python3.10/site-packages/peft/tuners/tuners_utils.py:167: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): PeftModelForCausalLM(
      (base_model): LoraModel(
        (model): Gemma2ForCausalLM(
          (model): Gemma2Model(
            (embed_tokens): Embedding(256000, 2304, padding_idx=0)
            (layers): ModuleList(
              (0-25): 26 x Gemma2DecoderLayer(
                (self_attn): Gemma2Attention(
                  (q_proj): lora.Linear4bit(
                    (base_layer): Linear4bit(in_features=2304, out_features=2048, bias=False)
                    (lora_dropout): ModuleDict(
                      (default): Identity()
                    )
                    (lora_A): ModuleDict(
                      (default): Linear(in_features=2304, out_features=16, bias=False)
                    )
                    (lora_B): ModuleDict(
                      (default): Linear(in_features=16, out_features=2048, bias=False)
                    )
                    (lora_embedding_A): ParameterDict()
            

In [ ]:
model.print_trainable_parameters()

trainable params: 6,389,760 || all params: 2,620,731,648 || trainable%: 0.2438


In [ ]:
test_generate(model, '한국의 수도는')

input : 한국의 수도는
output : 한국의 수도는 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울


In [ ]:
args = SFTConfig(
    output_dir = './qlora_output',
    num_train_epochs = 3,
    per_device_train_batch_size=2,
    learning_rate = 2e-5,
    bf16=True
)

trainer = SFTTrainer(
    model = model,
    args = args,
    train_dataset = dataset,
    processing_class = tokenizer)

trainer.train()

Adding EOS to train dataset:   0%|          | 0/8 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/8 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/8 [00:00<?, ? examples/s]

Step,Training Loss
10,4.524379


TrainOutput(global_step=12, training_loss=4.4297324021657305, metrics={'train_runtime': 5.5353, 'train_samples_per_second': 4.336, 'train_steps_per_second': 2.168, 'total_flos': 17376445836288.0, 'train_loss': 4.4297324021657305})

In [ ]:
test_generate(model, '한국의 수도는')

input : 한국의 수도는
output : 한국의 수도는 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울


In [ ]:
torch.cuda.memory_allocated() / 1024**3

23.767587184906006

In [ ]:
# del model
model_fp16 = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16)
torch.cuda.memory_allocated() / 1024**3

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

23.767587184906006

In [ ]:
# del model_fp16

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config = bnb_config, torch_dtype=torch.float16)

lora_config = LoraConfig(
    r=16, lora_alpha=32, target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'], task_type=TaskType.CAUSAL_LM)

model = get_peft_model(model, lora_config)

args = SFTConfig(
    output_dir = './qlora_output',
    num_train_epochs = 3,
    per_device_train_batch_size=2,
    learning_rate = 2e-5,
    bf16=True
)

trainer = SFTTrainer(
    model = model,
    args = args,
    train_dataset = dataset,
    processing_class = tokenizer)

trainer.train()

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

Adding EOS to train dataset:   0%|          | 0/8 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/8 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/8 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1}.


Step,Training Loss
10,4.530804


TrainOutput(global_step=12, training_loss=4.436998685201009, metrics={'train_runtime': 6.4368, 'train_samples_per_second': 3.729, 'train_steps_per_second': 1.864, 'total_flos': 17376445836288.0, 'train_loss': 4.436998685201009})

In [ ]:
test_generate(model, '한국의 수도는')

input : 한국의 수도는
output : 한국의 수도는 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울 서울


In [ ]:
model.save_pretrained('./my_lora_adapter')


In [ ]:
tokenizer.save_pretrained('./my_lora_adapter')

('./my_lora_adapter/tokenizer_config.json',
 './my_lora_adapter/chat_template.jinja',
 './my_lora_adapter/tokenizer.json')

In [ ]:
import os
os.listdir('./my_lora_adapter')

['adapter_model.safetensors',
 'tokenizer_config.json',
 'tokenizer.json',
 'adapter_config.json',
 'README.md',
 'chat_template.jinja']

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)

base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config = bnb_config, torch_dtype=torch.float16)

model2 = PeftModel.from_pretrained(base_model, './my_lora_adapter')

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

In [ ]:
test_generate(model2, '한국의 수도는')

input : 한국의 수도는
output : 한국의 수도는 서울입니다.

**서울**은 대한민국에서 가장 큰 도시 중 하나입니다. 

**서울의 역사와 문화**

* 서울은 역사적으로 매우 오래된 도시입니다. 
* 옛날에는 한반도의 중심지로서 중요한 역할을 했습니다. 
* 서울은 다양한 문화와 역사를 보


In [ ]:
merged_model = model2.merge_and_unload()

/home/oncreative/anaconda3/envs/codeit/lib/python3.10/site-packages/peft/tuners/lora/bnb.py:351: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


In [ ]:
merged_model

Gemma2ForCausalLM(
  (model): Gemma2Model(
    (embed_tokens): Embedding(256000, 2304, padding_idx=0)
    (layers): ModuleList(
      (0-25): 26 x Gemma2DecoderLayer(
        (self_attn): Gemma2Attention(
          (q_proj): Linear4bit(in_features=2304, out_features=2048, bias=False)
          (k_proj): Linear4bit(in_features=2304, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=2304, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=2048, out_features=2304, bias=False)
        )
        (mlp): Gemma2MLP(
          (gate_proj): Linear4bit(in_features=2304, out_features=9216, bias=False)
          (up_proj): Linear4bit(in_features=2304, out_features=9216, bias=False)
          (down_proj): Linear4bit(in_features=9216, out_features=2304, bias=False)
          (act_fn): GELUTanh()
        )
        (input_layernorm): Gemma2RMSNorm((2304,), eps=1e-06)
        (post_attention_layernorm): Gemma2RMSNorm((2304,), eps=1e-06)
        (pre_feedfor

In [ ]:
merged_model.save_pretrained('./merged_model')
tokenizer.save_pretrained('./merged_model')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./merged_model/tokenizer_config.json',
 './merged_model/chat_template.jinja',
 './merged_model/tokenizer.json')

In [ ]:
test_generate(merged_model, '한국의 수도는')

input : 한국의 수도는
output : 한국의 수도는 서울입니다.

**서울**은 대한민국에서 가장 큰 도시 중 하나입니다. 

**서울의 역사와 문화**

* 서울은 역사적으로 매우 오래된 도시입니다. 
* 옛날에는 한반도의 중심지로서 중요한 역할을 했습니다. 
* 서울은 다양한 문화와 역사를 보


In [ ]:
32bit -> 16bit
ㅇ, ㅇㅇㅇㅇㅇㅇㅇㅇㅇㅇ, ㅇㅇㅇㅇㅇㅇ, ㅇㅇㅇㅇ

In [ ]:
# unsigned
# float, int,  : signed (부호가 있다 +/-)
# ㅇㅇㅇㅇㅇㅇㅇㅇ : ㅇ  ㅇㅇㅇㅇㅇㅇㅇ   +/-  0000000 : 0   +/- 1111111

# ㅇㅇㅇㅇㅇㅇㅇㅇ : 00000000 ~ 11111111

# ㅇㅇㅇㅇ : ㅇ 0 000 ~ 0 111 : 0 ~ 7     1000 -7
#         ㅇ   000 ~ -111 :  -7 ~ 7

# 00000000 ~ 11111111 :
# 마이너스 : 2의 보수로 표현을 함
#  01010 ->  10110       10110
#    6   ->   -6



In [ ]:
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16).to(device)

inputs = tokenizer('한국의 수도는', return_tensors='pt').to(device)
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=60, do_sample=False)

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

In [ ]:
inputs['input_ids'].shape[1]

6

In [ ]:
out[0][inputs['input_ids'].shape[1]:]

tensor([163260,  47555, 235265,    109,    688, 211324,    688, 236648, 216101,
        237522,  22803, 126277, 185075, 204778,  47250, 113859, 225022, 235269,
         61169, 236417, 237807, 221628, 235269, 213512,  81386, 236800,  85024,
        204778, 236137,  42916, 236817, 236791,  11464, 238949, 236214, 204778,
         47555, 235265, 235248,    109,    688, 211324, 236137,  40712, 237526,
         55526, 239830,  95165, 237433,    688,    109, 235287,   5231, 237392,
        239205, 242383,  66058,  42916, 237700, 236569], device='cuda:0')

In [ ]:
tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

' 서울입니다.\n\n**서울**은 대한민국에서 가장 큰 도시 중 하나이며, 역사와 문화, 그리고 현대적인 도시의 조화를 이루는 도시입니다. \n\n**서울의 주요 관광 명소**\n\n* **경복궁:** 조선시'

In [ ]:
# model.config
sum(p.numel() for p in model.parameters())

2614341888

In [ ]:
config = model.config
config.hidden_size

2304

In [ ]:
config.num_hidden_layers

26

In [ ]:
config.num_attention_heads

8

In [ ]:
config.vocab_size

256000

In [ ]:
# for p in model.parameters():
#     dt = str(p.dtype)
#     print(dt)

# for name, module in model.named_modules():
#     if isinstance(module, torch.nn.Linear):
#         print(f" {name} : {module.in_features}, {module.out_features}")

In [ ]:
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16).to(device)

inputs = tokenizer('한국의 수도는', return_tensors='pt').to(device)
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=60, do_sample=False)
out

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

tensor([[     2, 236511, 142306,  22618, 236840, 236214, 163260,  47555, 235265,
            109,    688, 211324,    688, 236648, 216101, 237522,  22803, 126277,
         185075, 204778,  47250, 113859, 225022, 235269,  61169, 236417, 237807,
         221628, 235269, 213512,  81386, 236800,  85024, 204778, 236137,  42916,
         236817, 236791,  11464, 238949, 236214, 204778,  47555, 235265, 235248,
            109,    688, 211324, 236137,  40712, 237526,  55526, 239830,  95165,
         237433,    688,    109, 235287,   5231, 237392, 239205, 242383,  66058,
          42916, 237700, 236569]], device='cuda:0')

In [ ]:
def generate_text(prompt, max_new_tokens=100, do_sample=False, **kwargs):
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=do_sample, **kwargs)

    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

In [ ]:
prompt = "인공지능의 미래는"
result = generate_text(prompt, max_new_tokens=100)
print(result)

 끊임없이 변화하고 있으며, 이러한 변화는 우리의 삶에 큰 영향을 미치고 있습니다. 

**1. 혁신적인 기술 발전:**

* **자율주행:** 자동차, 로봇 등의 자율주행 기술은 급속도로 발전하고 있으며, 이는 교통 문제 해결, 안전성 향상, 그리고 새로운 산업 창출에 기여할 것으로 예


In [ ]:
from transformers import pipeline

In [ ]:
generator = pipeline('text-generation', model=model, tokenizer=tokenizer)
out = generator('인공지능의 미래는', max_new_tokens=100, do_sample=True)
print(out)

Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': '인공지능의 미래는 끊임없이 변화하고 있으며, 그 중 하나는 인공지능의 인간과의 관계에서의 새로운 패러다임을 제시하는 것으로 볼 수 있습니다.\n\n**1. 인공지능을 활용한 개인화된 서비스 제공:**\n\n* 예를 들어, 온라인 학습 플랫폼에서 학습자의 학습 스타일과 성취도를 분석하여 개별 맞춤형 학습 경험을 제공'}]


In [ ]:
prompt = "한국의 수도는"
inputs = tokenizer(prompt, return_tensors='pt').to(device)

In [ ]:
with torch.no_grad():
    outputs = model(**inputs)


In [ ]:
outputs.logits.shape

torch.Size([1, 6, 256000])

In [ ]:
next_token_logits = outputs.logits[0, -1, :]
next_token_logits.shape, next_token_logits.min(), next_token_logits.max()

(torch.Size([256000]),
 tensor(-26.3750, device='cuda:0', dtype=torch.float16),
 tensor(17.2812, device='cuda:0', dtype=torch.float16))

In [ ]:
import torch.nn.functional as F

for temp in [0.1, 0.5, 1.0, 2.0]:
    print(f"Temperature {temp}")
    probs = F.softmax(next_token_logits / temp, dim=-1)
    top_probs , top_ids = torch.topk(probs, 10)
    tokens = [tokenizer.decode([tid]) for tid in top_ids]
    for tok, prob in zip(tokens[:5], top_probs[:5]):
        print(f"{tok} : {prob}")

    top1_prob = top_probs[0].item()
    top5_prob = top_probs[:5].sum().item()
    print(f"top1 prob : {top1_prob}, top5 prob : {top5_prob}")
    print("\n\n")

Temperature 0.1
 서울 : 0.99951171875
 ** : 0.0005526542663574219
<pad> : 0.0
<eos> : 0.0
[@BOS@] : 0.0
top1 prob : 0.99951171875, top5 prob : 1.0



Temperature 0.5
 서울 : 0.76806640625
 ** : 0.17138671875
 부 : 0.018646240234375
 어 : 0.0157012939453125
  : 0.0107879638671875
top1 prob : 0.76806640625, top5 prob : 0.984375



Temperature 1.0
 서울 : 0.4033203125
 ** : 0.1905517578125
 부 : 0.06280517578125
 어 : 0.057647705078125
  : 0.04779052734375
top1 prob : 0.4033203125, top5 prob : 0.76220703125



Temperature 2.0
 서울 : 0.05230712890625
 ** : 0.03594970703125
 부 : 0.0206451416015625
 어 : 0.019775390625
  : 0.01800537109375
top1 prob : 0.05230712890625, top5 prob : 0.146728515625





In [ ]:
# Quantized LoRA

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type = "nf4",
    bnb_4bit_compute_dtype = torch.float16,
    bnb_4bit_use_double_quant = True
)

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config =bnb_config, )

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]